# Assignment 1 — EDA and Baseline Models
**CENG313 Data Science · Group 53**

**Competition:** [Rohlik Sales Forecasting Challenge v2](https://www.kaggle.com/competitions/rohlik-sales-forecasting-challenge-v2)

**Task:** forecast the daily `sales` of each grocery item (`unique_id`) in each Rohlik warehouse for the 14 days that follow the training period.

**Metric:** Weighted Mean Absolute Error (WMAE), with one weight per item from `test_weights.csv`. Lower is better.

### Contents
1. Setup
2. Loading the data
3. Data quality checks
4. Target variable (`sales`)
5. Trends over time
6. Seasonality
7. Holidays and the calendar
8. Prices and discounts
9. Availability
10. Product categories
11. Item-level behaviour and evaluation weights
12. Correlations
13. Train vs. test
14. EDA summary
15. Baseline models
16. Kaggle submission
17. Conclusion

## 1. Setup
On Colab this cell clones the repository and downloads the data with the Kaggle API. Upload your `kaggle.json` (Kaggle → Settings → API → *Create New Token*) when asked. The competition rules must be accepted on Kaggle first.

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AliOsmanBltt/GROUP53_CENG313.git"

if IN_COLAB:
    if not os.path.exists("GROUP53_CENG313"):
        !git clone -q {REPO_URL}
    %cd GROUP53_CENG313
    if not os.path.exists(os.path.expanduser("~/.kaggle/kaggle.json")):
        from google.colab import files
        files.upload()  # select kaggle.json
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
    !pip install -q kaggle

from src.data import download_data
download_data("data")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import data as D
from src import models as M
from src import utils as U

U.set_seed(42)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (12, 4)

## 2. Loading the data
The competition provides five files.

In [ ]:
raw = D.load_raw("data")
for name, df in raw.items():
    print(f"{name:<10} {df.shape[0]:>10,} rows × {df.shape[1]:>2} columns")

In [ ]:
raw['train'].head()

In [ ]:
raw['test'].head()

In [ ]:
raw['inventory'].head()

In [ ]:
raw['calendar'].head()

In [ ]:
raw['weights'].head()

In [ ]:
raw['train'].info(memory_usage='deep')

## 3. Data quality checks

In [ ]:
train_raw, test_raw = raw["train"], raw["test"]
print("Train dates:", train_raw["date"].min().date(), "→", train_raw["date"].max().date())
print("Test dates: ", test_raw["date"].min().date(), "→", test_raw["date"].max().date())
print("Unique items in train:", train_raw["unique_id"].nunique())
print("Unique items in test: ", test_raw["unique_id"].nunique())
print("Test items without train history:",
      len(set(test_raw["unique_id"]) - set(train_raw["unique_id"])))
print("Duplicate (unique_id, date) rows in train:",
      train_raw.duplicated(["unique_id", "date"]).sum())
print("Warehouses:", sorted(train_raw["warehouse"].unique()))

In [ ]:
print("Missing values in train:")
display(U.missing_report(train_raw))
print("Missing values in test:")
display(U.missing_report(test_raw))

In [ ]:
# discount columns: check the value range before cleaning
train_raw[D.DISCOUNT_COLS].describe().T

In [ ]:
# How complete is each item's history? (observed days / days between first and last sale)
span = train_raw.groupby("unique_id")["date"].agg(["min", "max", "count"])
span["days"] = (span["max"] - span["min"]).dt.days + 1
span["coverage"] = span["count"] / span["days"]
display(span["coverage"].describe())
span["coverage"].hist(bins=50)
plt.title("Share of calendar days with a record, per item")
plt.xlabel("coverage"); plt.ylabel("items")
plt.show()

Next we clean the data (drop rows without `sales`, clip discounts to [0, 1], add `max_discount`) and join item info, calendar flags and weights. The code for this lives in `src/data.py`.

In [ ]:
train, test = D.prepare(raw)
train = D.reduce_memory(train)
print(train.shape, test.shape)
train.head()

## 4. Target variable (`sales`)

In [ ]:
display(train["sales"].describe(percentiles=[.01, .25, .5, .75, .9, .99]).to_frame().T)
print(f"Share of zero sales: {(train['sales'] == 0).mean():.2%}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
train["sales"].clip(upper=train["sales"].quantile(.99)).hist(bins=100, ax=axes[0])
axes[0].set_title("Sales (clipped at 99th percentile)")
np.log1p(train["sales"]).hist(bins=100, ax=axes[1])
axes[1].set_title("log1p(sales)")
U.save_fig(fig, "01_sales_distribution")
plt.show()

## 5. Trends over time

In [ ]:
daily = train.groupby("date").agg(sales=("sales", "sum"), items=("unique_id", "nunique"))
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
daily["sales"].plot(ax=axes[0], lw=0.6)
daily["sales"].rolling(28).mean().plot(ax=axes[0], lw=2, label="28-day mean")
axes[0].set_title("Total daily sales"); axes[0].legend()
daily["items"].plot(ax=axes[1], lw=0.8)
axes[1].set_title("Number of active items per day")
U.save_fig(fig, "02_daily_trend")
plt.show()

In [ ]:
wh_daily = train.groupby(["date", "warehouse"])["sales"].sum().unstack()
ax = wh_daily.rolling(28).mean().plot(figsize=(14, 5), lw=1.2)
ax.set_title("Daily sales per warehouse (28-day rolling mean)")
U.save_fig(ax.figure, "03_warehouse_trend")
plt.show()

In [ ]:
# total_orders is known in advance (also in the test set) — how does it relate to sales?
wh = train.groupby(["date", "warehouse"]).agg(sales=("sales", "sum"),
                                               orders=("total_orders", "first")).reset_index()
ax = sns.scatterplot(data=wh, x="orders", y="sales", hue="warehouse", s=6, alpha=.5)
ax.set_title("Warehouse daily total_orders vs. total sales")
plt.show()
print("Correlation:", wh[["orders", "sales"]].corr().iloc[0, 1].round(3))

## 6. Seasonality

In [ ]:
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
dow = train.groupby("dayofweek")["sales"].mean()
axes[0].bar(days, dow.values)
axes[0].set_title("Mean sales per item by weekday")
month = train.groupby("month")["sales"].mean()
axes[1].bar(month.index, month.values)
axes[1].set_title("Mean sales per item by month"); axes[1].set_xticks(range(1, 13))
U.save_fig(fig, "04_seasonality")
plt.show()

In [ ]:
# weekday profile per warehouse, normalised by the warehouse mean
prof = train.pivot_table(index="warehouse", columns="dayofweek", values="sales", aggfunc="mean")
prof = prof.div(prof.mean(axis=1), axis=0)
prof.columns = days
ax = sns.heatmap(prof, annot=True, fmt=".2f", cmap="RdBu_r", center=1)
ax.set_title("Weekday index per warehouse (1 = warehouse average)")
plt.show()

## 7. Holidays and the calendar

In [ ]:
for flag in D.CALENDAR_FLAGS:
    g = train.groupby(flag)["sales"].agg(["mean", "size"])
    print(f"{flag:<24} mean sales: no={g.loc[0, 'mean']:.1f}  "
          f"yes={g['mean'].get(1, np.nan):.1f}   (rows with flag: {g['size'].get(1, 0):,})")

In [ ]:
# daily total sales in the week around each public holiday (Prague_1 as an example)
cal = raw["calendar"]
wh_name = "Prague_1"
s = train[train["warehouse"] == wh_name].groupby("date")["sales"].sum()
hol = cal[(cal["warehouse"] == wh_name) & (cal["holiday"] == 1) &
          (cal["date"].between(s.index.min(), s.index.max()))]["date"]
rows = []
for h in hol:
    for k in range(-7, 8):
        d = h + pd.Timedelta(days=k)
        if d in s.index:
            rows.append({"offset": k, "ratio": s[d] / s.rolling(28, min_periods=7).mean().get(d, np.nan)})
eff = pd.DataFrame(rows).groupby("offset")["ratio"].mean()
ax = eff.plot(marker="o")
ax.axvline(0, color="red", ls="--")
ax.set_title(f"{wh_name}: sales relative to 28-day mean around holidays (0 = holiday)")
ax.set_ylabel("sales / rolling mean")
U.save_fig(ax.figure, "05_holiday_effect")
plt.show()

## 8. Prices and discounts

In [ ]:
print(f"Rows with any discount: {(train['max_discount'] > 0).mean():.2%}")
print("Share of rows where each discount type is active:")
display((train[D.DISCOUNT_COLS] > 0).mean().round(4).to_frame("share").T)

bins = [-0.001, 0, 0.1, 0.2, 0.3, 0.4, 0.5, 1]
labels = ["0%", "0-10%", "10-20%", "20-30%", "30-40%", "40-50%", ">50%"]
train["discount_bucket"] = pd.cut(train["max_discount"], bins=bins, labels=labels)

# compare each item's sales with its own average to remove the item-size effect
item_mean = train.groupby("unique_id")["sales"].transform("mean")
train["sales_rel"] = train["sales"] / item_mean.replace(0, np.nan)
lift = train.groupby("discount_bucket", observed=True)["sales_rel"].mean()
ax = lift.plot(kind="bar")
ax.axhline(1, color="red", ls="--")
ax.set_title("Sales relative to the item's own mean, by discount depth")
U.save_fig(ax.figure, "06_discount_lift")
plt.show()

In [ ]:
ax = np.log1p(train["sell_price_main"]).hist(bins=80)
ax.set_title("log1p(sell_price_main)")
plt.show()

## 9. Availability
`availability` exists only in train, so it cannot be used directly as a feature for test.

In [ ]:
display(train["availability"].describe().to_frame().T)
print(f"Rows with availability < 1: {(train['availability'] < 1).mean():.2%}")
avail_bins = pd.cut(train["availability"], [0, .5, .8, .9, .99, 1.0])
ax = train.groupby(avail_bins, observed=True)["sales_rel"].mean().plot(kind="bar")
ax.set_title("Relative sales by availability bucket")
plt.show()

## 10. Product categories

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
l1 = train.groupby("L1_category_name_en")["sales"].sum().sort_values()
(l1 / l1.sum()).plot(kind="barh", ax=axes[0])
axes[0].set_title("Share of total sales by L1 category")
l2 = train.groupby("L2_category_name_en")["sales"].sum().nlargest(15).sort_values()
(l2 / train["sales"].sum()).plot(kind="barh", ax=axes[1])
axes[1].set_title("Top 15 L2 categories by share of sales")
U.save_fig(fig, "07_categories")
plt.show()
print("Distinct categories per level:",
      {c: raw["inventory"][c].nunique() for c in D.CATEGORY_COLS})

## 11. Item-level behaviour and evaluation weights

In [ ]:
item = train.groupby("unique_id").agg(mean_sales=("sales", "mean"), cv=("sales", "std"),
                                      weight=("weight", "first"), n_days=("sales", "size"))
item["cv"] = item["cv"] / item["mean_sales"].replace(0, np.nan)
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
np.log1p(item["mean_sales"]).hist(bins=60, ax=axes[0]); axes[0].set_title("log1p(mean sales per item)")
item["weight"].hist(bins=60, ax=axes[1]); axes[1].set_title("Evaluation weight per item")
axes[2].scatter(item["mean_sales"], item["weight"], s=3, alpha=.4)
axes[2].set_xscale("log"); axes[2].set_xlabel("mean sales"); axes[2].set_ylabel("weight")
axes[2].set_title("Weight vs. mean sales")
U.save_fig(fig, "08_items_weights")
plt.show()
print("Spearman corr(weight, mean sales):",
      item[["weight", "mean_sales"]].corr(method="spearman").iloc[0, 1].round(3))

In [ ]:
# a few example series
sample_ids = item.sort_values("weight", ascending=False).index[:3]
fig, axes = plt.subplots(3, 1, figsize=(14, 8))
for ax, uid in zip(axes, sample_ids):
    U.plot_item_series(train, uid, ax=ax)
plt.tight_layout()
plt.show()

## 12. Correlations

In [ ]:
num_cols = ["sales", "total_orders", "sell_price_main", "max_discount", "availability",
            "holiday", "shops_closed", "school_holidays", "dayofweek", "month"]
ax = sns.heatmap(train[num_cols].corr(method="spearman"), annot=True, fmt=".2f",
                 cmap="RdBu_r", center=0)
ax.set_title("Spearman correlation")
plt.show()

## 13. Train vs. test

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for df, lab in [(train[train["date"] > train["date"].max() - pd.Timedelta(days=28)], "train (last 28 days)"),
                (test, "test")]:
    axes[0].hist(np.log1p(df["sell_price_main"].dropna()), bins=50, density=True, alpha=.5, label=lab)
    axes[1].hist(df["max_discount"], bins=np.linspace(0, 1, 41), density=True, alpha=.5, label=lab)
axes[0].set_title("log1p(price)"); axes[1].set_title("max_discount")
axes[0].legend(); axes[1].legend()
plt.show()
print("Test rows per warehouse:")
display(test["warehouse"].value_counts().to_frame().T)

## 14. EDA summary
> **TODO:** update these points with the exact numbers printed above after running the notebook.

- **Problem type:** each test item already has history in train, so this is a "continue the series" problem. The horizon is 14 days.
- **Target:** `sales` is strongly right-skewed and its scale differs a lot between items, so per-item statistics matter more than global ones.
- **Trend:** total sales and the number of active items grow over time. Warehouses differ a lot in volume.
- **Seasonality:** there is a clear weekly pattern, which is why weekday-aware baselines are included.
- **Calendar:** holidays and `shops_closed` days change sales noticeably, and the flags are known for the test period.
- **Discounts:** deeper discounts go together with higher sales relative to the item's own mean.
- **Known-in-advance features:** `total_orders`, `sell_price_main` and the discount columns are given for the test dates. `availability` is not.
- **Weights:** evaluation weights are skewed, so a few items dominate WMAE.

## 15. Baseline models
**Validation setup.** The last 14 days of train are held out as a validation set, mirroring the 14-day Kaggle test horizon. Every model is trained on the earlier data and scored with WMAE on that window.

| Baseline | Idea |
|---|---|
| Naive (last value) | last observed sales of the item |
| Seasonal naive | last observed sales on the same weekday |
| Moving average (7d / 28d) | mean of the last 7 / 28 days |
| Weekday mean (8w) | the item's mean per weekday over the last 8 weeks |
| Ridge regression | linear model on calendar, price, discount and item-level statistics (log target) |
| Random forest | same features, non-linear model |

All baselines are implemented in `src/models.py` with a common `fit` / `predict` interface.

In [ ]:
tr, va = D.time_split(train, horizon_days=14)
print("train:", tr["date"].min().date(), "→", tr["date"].max().date(), f"({len(tr):,} rows)")
print("valid:", va["date"].min().date(), "→", va["date"].max().date(), f"({len(va):,} rows)")

In [ ]:
baselines = M.all_baselines(seed=42)
results, val_preds = U.evaluate_models(baselines, tr, va)
results

In [ ]:
ax = results.set_index("model")["WMAE"].sort_values().plot(kind="barh")
ax.set_title("Validation WMAE of the baselines (lower is better)")
U.save_fig(ax.figure, "09_baseline_wmae")
plt.show()

In [ ]:
# where does the best baseline make its errors?
best = results.loc[0, "model"]
err = va.assign(abs_err=np.abs(va["sales"] - val_preds[best]) * va["weight"])
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(err.groupby("warehouse")["abs_err"].sum() / err.groupby("warehouse")["weight"].sum()).sort_values().plot(
    kind="barh", ax=axes[0], title=f"{best}: WMAE per warehouse")
(err.groupby("date")["abs_err"].sum() / err.groupby("date")["weight"].sum()).plot(
    ax=axes[1], marker="o", title=f"{best}: WMAE per validation day")
plt.show()

In [ ]:
# predictions vs. actuals for the highest-weight items
show = {k: val_preds[k] for k in [best, "Seasonal naive (same weekday)"] if k in val_preds}
fig, axes = plt.subplots(3, 1, figsize=(14, 9))
for ax, uid in zip(axes, sample_ids):
    U.plot_item_series(tr[tr["date"] > tr["date"].max() - pd.Timedelta(days=60)], uid,
                       ax=ax, valid=va, preds=show)
plt.tight_layout()
plt.show()

In [ ]:
# which features does the random forest rely on?
rf = next(m for m in baselines if m.name == "Random forest")
imp = pd.Series(rf.estimator.feature_importances_, index=rf.feature_names_).nlargest(15).sort_values()
ax = imp.plot(kind="barh", title="Random forest feature importance (top 15)")
plt.show()

## 16. Kaggle submission
The best baseline is refitted on the full training data and used to predict the test set. The competition has ended, but late submissions are still scored. Upload the file under *Late Submission* and record the public/private scores in the README.

In [ ]:
best_model = next(m for m in M.all_baselines(seed=42) if m.name == best)
best_model.fit(train)
test_pred = best_model.predict(test)
slug = best.lower().replace(" ", "_").replace("(", "").replace(")", "")
sub = U.make_submission(test, test_pred, f"submissions/a1_{slug}.csv")
sub.head()

In [ ]:
# optional: submit directly from the notebook
# !kaggle competitions submit -c rohlik-sales-forecasting-challenge-v2 -f submissions/<file>.csv -m "A1 baseline"

## 17. Conclusion
> **TODO:** fill in after running.

- Best baseline on validation: **…** (WMAE = …).
- Kaggle late-submission score: public … / private ….
- **Ideas for Assignment 2:** lag and rolling-window features, explicit use of `total_orders` and the discount features, gradient boosting (LightGBM / XGBoost) trained with the evaluation weights, a direct multi-horizon setup, and ensembling.